In [ ]:

from pathlib import Path
import pandas as pd

# Project root directory
PROJECT_DIR = Path.cwd().parent

# Standardized sensor directory
DATA_DIR = (
    PROJECT_DIR
    / "data"
    / "interim"
    / "standardized"
    / "sensors"
)

print("Project directory:", PROJECT_DIR)
print("Sensor directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

if DATA_DIR.exists():

    files = sorted(DATA_DIR.glob("*.parquet"))

    print("\nParquet files found:", len(files))

    for file in files:
        print(" -", file.name)

else:
    print("\nSensor directory not found. Check folder structure.")


In [ ]:

import pandas as pd

sensor_files = sorted(DATA_DIR.glob("*.parquet"))

for file_path in sensor_files:

    df = pd.read_parquet(file_path)

    print("\n" + "=" * 70)
    print(f"SENSOR: {file_path.stem}")
    print("=" * 70)

    print("Total records:", len(df))
    print("Total columns:", len(df.columns))

    print("\nCOLUMN NAMES AND DATA TYPES:")

    for col in df.columns:
        print(f"{col:35} {df[col].dtype}")

    print("\nMISSING VALUES:")

    missing = df.isnull().sum()
    print(missing[missing > 0])

    print("\nDUPLICATE ROWS:", df.duplicated().sum())

    print("\nFIRST 2 ROWS (excluding participant IDs):")

    display(
        df.drop(
            columns=["participant_id"],
            errors="ignore"
        ).head(2)
    )


In [ ]:

# STEP 3: Detailed validation of standardized sensor data

import pandas as pd

validation_results = []

for file_path in sorted(DATA_DIR.glob("*.parquet")):

    df = pd.read_parquet(file_path)

    sensor = file_path.stem

    print("\n" + "=" * 65)
    print(f"VALIDATING: {sensor}")
    print("=" * 65)

    # Basic statistics
    print("Total records:", len(df))
    print("Unique participants:", df["Participant_ID"].nunique())
    print("Unique timestamps:", df["Timestamp"].nunique())
    print("Exact duplicate rows:", df.duplicated().sum())

    # Missing values
    missing_count = int(df.isna().sum().sum())
    print("Total missing values:", missing_count)

    # Timestamp validation
    print("Missing timestamps:", df["Timestamp"].isna().sum())
    print("Timestamp timezone:", df["Timestamp"].dt.tz)

    # Validity information
    for col in ["index_validity", "data_validity"]:
        if col in df.columns:
            print(f"\n{col} distribution:")
            print(df[col].value_counts(dropna=False).to_dict())

    # Participant and country consistency
    print("\nCountries:", df["Country"].dropna().unique())

    validation_results.append({
        "sensor": sensor,
        "records": len(df),
        "participants": df["Participant_ID"].nunique(),
        "missing_values": missing_count,
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_timestamps": int(df["Timestamp"].isna().sum())
    })

# Summary table
validation_df = pd.DataFrame(validation_results)

print("\n\nFINAL VALIDATION SUMMARY")
display(validation_df)

# Inspect other standardized Parquet files
STANDARDIZED_DIR = PROJECT_DIR / "data" / "interim" / "standardized"

print("\nALL OTHER STANDARDIZED PARQUET FILES:")

for file in sorted(STANDARDIZED_DIR.rglob("*.parquet")):
    if file.parent != DATA_DIR:
        print(file.relative_to(PROJECT_DIR))



## Step 3: Detailed Sensor Data Validation

### Objective
Validate the standardized sensor datasets before database ingestion.

### Checks performed
1. Total records and unique participants.
2. Missing values in each sensor dataset.
3. Duplicate rows.
4. Timestamp completeness and timezone.
5. Distribution of index_validity and data_validity.
6. Country information.
7. Availability of other standardized Parquet files.

### Data integrity decisions
- Original sensor records will be preserved.
- Missing values will not be replaced with zero.
- Duplicate timestamps will not automatically be removed.
- Validity flags will be preserved.
- All timestamps will retain their UTC interpretation.

### Results
To be completed after validation.


In [ ]:

# Find all standardized Parquet files and dataset documentation

print("OTHER STANDARDIZED PARQUET FILES")
print("=" * 60)

all_parquet = list(STANDARDIZED_DIR.rglob("*.parquet"))

for file in sorted(all_parquet):
    if file.parent != DATA_DIR:
        print(file.relative_to(PROJECT_DIR))

print("\nDATASET README FILES")
print("=" * 60)

for file in PROJECT_DIR.rglob("*"):
    if (
        file.is_file()
        and file.name.lower().startswith("readme")
        and ".git" not in file.parts
    ):
        print(file.relative_to(PROJECT_DIR))



## Step 3: Detailed Sensor Data Validation

### Objective
Validate the eight standardized sensor datasets before importing them into Supabase PostgreSQL.

### Observed Results
- Total sensor records: 91,794.
- Unique participants across sensor datasets: 74.
- Countries represented: Germany and Spain.
- Missing timestamps: 0 across all eight datasets.
- Exact duplicate rows were observed in EI, PRVI and SII.
- Missing measurement and validity values were observed in BRI, PEI, PRVI and SQI.

### Data Integrity Decisions
1. Preserve original sensor records, including exact duplicate rows.
2. Preserve missing values as NULL.
3. Preserve UTC timestamps.
4. Preserve original validity flags without assuming their meaning.
5. Use deterministic source record identifiers for repeatable ingestion.
6. Do not modify standardized Parquet files.

### Interpretation
The standardized sensor files are readable and contain the expected eight sensor streams. Additional documentation review is required before interpreting validity flags and importing clinical information.

### Next Step
Inspect dataset documentation and remaining standardized Parquet files before configuring database ingestion.


In [ ]:
README_PATH = PROJECT_DIR / "README.md"

if README_PATH.exists():
    readme_text = README_PATH.read_text(encoding="utf-8")
    print(readme_text)
else:
    print("README not found at:", README_PATH)

In [ ]:
DATA_ROOT = PROJECT_DIR / "data"

for folder in ["raw", "interim", "processed"]:
    path = DATA_ROOT / folder

    print(f"\n--- {folder.upper()} ---")

    if path.exists():
        for file in sorted(path.rglob("*")):
            if file.is_file() and file.suffix.lower() in [
                ".parquet", ".csv", ".xlsx", ".json"
            ]:
                print(file.relative_to(PROJECT_DIR))

In [ ]:
from pathlib import Path
import pandas as pd
from collections import Counter

RAW_INDEX_DIR = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "TOLIFE_COPD_Dataset"
    / "index"
)

clinical_files = sorted(RAW_INDEX_DIR.rglob("clinical_indices.csv"))
exacerbation_files = sorted(RAW_INDEX_DIR.rglob("exacerbations.csv"))

print("CLINICAL FILES:", len(clinical_files))
print("EXACERBATION FILES:", len(exacerbation_files))


def inspect_csv_collection(files, label):
    print(f"\n{'=' * 65}")
    print(label)
    print("=" * 65)

    schema_counts = Counter()

    for file in files:
        try:
            df = pd.read_csv(file)
            schema = tuple(df.columns)
            schema_counts[schema] += 1
        except Exception as e:
            print(f"Could not read {file}: {e}")

    print("\nSchema variations:")
    for schema, count in schema_counts.items():
        print(f"\nFound in {count} file(s)")
        print("Columns:", list(schema))

    if files:
        print("\nFirst file preview:")
        sample = pd.read_csv(files[0])
        print("File:", files[0].relative_to(PROJECT_DIR))
        print("Shape:", sample.shape)
        display(sample.head())
        print("\nData types:")
        print(sample.dtypes)


inspect_csv_collection(clinical_files, "CLINICAL ASSESSMENTS")
inspect_csv_collection(exacerbation_files, "EXACERBATION EVENTS")

## Step 5: Inspection of Clinical and Exacerbation Data

### Objective
Identify the raw clinical assessment and exacerbation event files available in the TOLIFE dataset before designing their database ingestion process.

### Dataset File Inventory
- Clinical assessment files: 74 `clinical_indices.csv` files.
- Exacerbation event files: 24 `exacerbations.csv` files.
- Both file types are located within individual participant folders under the raw dataset directory.

### Inspection Approach
The CSV files are inspected for:
- Column names and schema consistency.
- Record counts and sample observations.
- Data types.
- Fields relevant to clinical assessments and exacerbation events.

### Data Integrity
The raw CSV files are read-only inputs. No modifications are made during this inspection.

### Next Step
Review the observed schemas and determine how clinical assessment records and exacerbation events should map to the existing PostgreSQL tables.

### Step 5: Inspect Clinical Assessment and Exacerbation Schemas

**Objective:** Inspect the raw clinical assessment and exacerbation CSV files before designing their database import process.

**Observed results:**

- Found 74 `clinical_indices.csv` files.
- All 74 clinical files share the same 35-column schema.
- Each clinical file contains five timepoint labels: T1, T2, T3, T4, and T5.
- The clinical dataset includes demographic information, lung-function measurements, symptom scores, blood pressure, oxygen saturation, sleep measurements, quality-of-life scores, and walking distance.
- Found 24 `exacerbations.csv` files.
- All 24 exacerbation files share the same four-column schema: `Date`, `PatientId`, `Kit-id`, and `Type`.

**Planned database mapping:**

| Source information | Database destination |
|---|---|
| Patient-ID / PatientId | Resolve to the corresponding patient record |
| Timepoint | `clinical_assessments.visit_type` |
| Visit Date | `clinical_assessments.visit_date` |
| Clinical measurements | `clinical_assessments.clinical_data` (JSONB) |
| Date | `exacerbations.event_timestamp` |
| Type | `exacerbations.severity` |
| Kit-id | Preserve in event metadata |

**Important data-handling decisions:**

1. Preserve original clinical measurement names inside the JSONB field.
2. Do not insert entirely empty clinical timepoint rows as completed assessments.
3. Preserve the original Kit-id when importing exacerbation events.
4. Verify patient identifiers across the raw and standardized datasets before linking records.

**Interpretation:** Both datasets have consistent file schemas, which makes a common import process possible. However, schema consistency does not guarantee that every row contains a completed assessment or that patient identifiers match across datasets. These checks are required before database insertion.

In [ ]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Collect clinical assessment information
# --------------------------------------------------

clinical_records = []

for file in clinical_files:
    df = pd.read_csv(file)

    kit_id = file.parent.name

    for _, row in df.iterrows():
        clinical_records.append({
            "kit_id": kit_id,
            "patient_id": str(row["Patient-ID"]),
            "timepoint": row["Timepoint"],
            "visit_date": row["Visit Date"],
            "has_measurements": row.drop(
                labels=["Patient-ID", "Timepoint", "Visit Date"],
                errors="ignore"
            ).notna().any()
        })

clinical_summary = pd.DataFrame(clinical_records)

print("CLINICAL RECORD SUMMARY")
print("=" * 50)
print("Total rows:", len(clinical_summary))
print("Rows with at least one clinical measurement:",
      clinical_summary["has_measurements"].sum())
print("Rows without clinical measurements:",
      (~clinical_summary["has_measurements"]).sum())

print("\nRows with visit dates:")
print(clinical_summary["visit_date"].notna().sum())

print("\nTimepoint distribution:")
print(clinical_summary["timepoint"].value_counts().sort_index())

print("\nClinical patient IDs:", clinical_summary["patient_id"].nunique())
print("Clinical kit IDs:", clinical_summary["kit_id"].nunique())


# --------------------------------------------------
# 2. Collect exacerbation information
# --------------------------------------------------

exacerbation_records = []

for file in exacerbation_files:
    df = pd.read_csv(file)

    for _, row in df.iterrows():
        exacerbation_records.append({
            "patient_id": str(row["PatientId"]),
            "kit_id": str(row["Kit-id"]),
            "date": row["Date"],
            "severity": row["Type"]
        })

exacerbation_summary = pd.DataFrame(exacerbation_records)

print("\n\nEXACERBATION RECORD SUMMARY")
print("=" * 50)
print("Total events:", len(exacerbation_summary))
print("Unique patient IDs:",
      exacerbation_summary["patient_id"].nunique())

print("\nSeverity distribution:")
print(exacerbation_summary["severity"].value_counts(dropna=False))

print("\nMissing dates:",
      exacerbation_summary["date"].isna().sum())

print("\nMissing patient IDs:",
      exacerbation_summary["patient_id"].isna().sum())


# --------------------------------------------------
# 3. Compare identifiers with standardized sensors
# --------------------------------------------------

sensor_participants = set()

for file in sorted(DATA_DIR.glob("*.parquet")):
    df = pd.read_parquet(file)
    sensor_participants.update(
        df["Participant_ID"].dropna().astype(str).unique()
    )

clinical_patient_ids = set(clinical_summary["patient_id"])
clinical_kit_ids = set(clinical_summary["kit_id"])

exacerbation_patient_ids = set(exacerbation_summary["patient_id"])
exacerbation_kit_ids = set(exacerbation_summary["kit_id"])

print("\n\nIDENTIFIER COMPARISON")
print("=" * 50)

print("Example sensor Participant_ID values:",
      sorted(sensor_participants)[:10])

print("\nExample clinical Patient-ID values:",
      sorted(clinical_patient_ids)[:10])

print("\nExample clinical kit IDs:",
      sorted(clinical_kit_ids)[:10])

print("\nClinical Patient-ID matches sensor Participant_ID:",
      len(clinical_patient_ids & sensor_participants))

print("Clinical kit IDs match sensor Participant_ID:",
      len(clinical_kit_ids & sensor_participants))

print("\nExacerbation PatientId matches sensor Participant_ID:",
      len(exacerbation_patient_ids & sensor_participants))

print("Exacerbation Kit-id matches sensor Participant_ID:",
      len(exacerbation_kit_ids & sensor_participants))

### Step 6: Clinical and Exacerbation Record Validation

**Objective:** Determine how clinical assessments and exacerbation events relate to the standardized sensor participant records.

**Observed clinical results:**

- Total timepoint rows: 370.
- Rows containing at least one clinical measurement: 224.
- Entirely empty rows: 146.
- Rows with visit dates: 224.
- Unique clinical Patient-ID values: 74.
- Unique clinical kit IDs: 74.
- Each participant has five timepoint rows (T1–T5).

**Observed exacerbation results:**

- Total events: 39.
- Unique patients: 24.
- Moderate events: 29.
- Severe events: 10.
- Missing dates: 0.
- Missing patient IDs: 0.

**Identifier reconciliation:**

The standardized sensor files use `Participant_ID` values such as `GER-001`. Clinical files use numeric `Patient-ID` values, while exacerbation files use numeric `PatientId` values.

The clinical `Kit ID` and exacerbation `Kit-id` values match the standardized sensor `Participant_ID` values:

- Clinical kit IDs matching sensor participant IDs: 74.
- Exacerbation kit IDs matching sensor participant IDs: 24.

**Database linking strategy:**

Use the kit identifier to resolve each source record to the corresponding patient in the `patients` table. Then use the database-generated `patients.id` as the foreign key.

**Import decisions:**

1. Import the 224 clinical rows containing measurements, rather than all 370 timepoint rows.
2. Preserve the original numeric patient identifiers in the clinical and event metadata.
3. Preserve the original kit identifiers.
4. Import all 39 exacerbation events after validating their patient relationships.

**Interpretation:** The kit identifier provides a common participant reference across the sensor, clinical, and exacerbation datasets. This allows the records to be linked without assuming that their numeric patient identifiers are the same.

In [ ]:
import pandas as pd

# --------------------------------------------------
# 1. Build sensor participant-country mapping
# --------------------------------------------------

sensor_mapping_frames = []

for file in sorted(DATA_DIR.glob("*.parquet")):
    df = pd.read_parquet(file)

    mapping = df[["Participant_ID", "Country"]].drop_duplicates()
    sensor_mapping_frames.append(mapping)

sensor_mapping = pd.concat(
    sensor_mapping_frames,
    ignore_index=True
).drop_duplicates()

sensor_mapping["Participant_ID"] = (
    sensor_mapping["Participant_ID"].astype(str)
)

print("SENSOR MAPPING")
print("=" * 50)
print("Unique participants:",
      sensor_mapping["Participant_ID"].nunique())

print("Participants with multiple country values:",
      (sensor_mapping.groupby("Participant_ID")["Country"]
       .nunique() > 1).sum())


# --------------------------------------------------
# 2. Build clinical kit-to-patient mapping
# --------------------------------------------------

clinical_mapping_records = []

for file in clinical_files:
    df = pd.read_csv(file)

    clinical_mapping_records.append({
        "kit_id": file.parent.name,
        "clinical_patient_id": str(df["Patient-ID"].iloc[0]),
        "clinical_patient_id_count": df["Patient-ID"].nunique()
    })

clinical_mapping = pd.DataFrame(clinical_mapping_records)

print("\nCLINICAL MAPPING")
print("=" * 50)
print("Unique kit IDs:", clinical_mapping["kit_id"].nunique())

print("Files with multiple clinical Patient-ID values:",
      (clinical_mapping["clinical_patient_id_count"] > 1).sum())


# --------------------------------------------------
# 3. Build exacerbation kit-to-patient mapping
# --------------------------------------------------

exacerbation_mapping = (
    exacerbation_summary[
        ["kit_id", "patient_id"]
    ]
    .drop_duplicates()
    .rename(columns={"patient_id": "exacerbation_patient_id"})
)

events_per_kit = (
    exacerbation_summary.groupby("kit_id")
    .size()
    .rename("event_count")
    .reset_index()
)

exacerbation_mapping = exacerbation_mapping.merge(
    events_per_kit,
    on="kit_id",
    how="left"
)

print("\nEXACERBATION MAPPING")
print("=" * 50)
print("Unique kit IDs:",
      exacerbation_mapping["kit_id"].nunique())

print("Kits associated with multiple numeric patient IDs:",
      (exacerbation_mapping.groupby("kit_id")
       ["exacerbation_patient_id"].nunique() > 1).sum())


# --------------------------------------------------
# 4. Combine into one crosswalk
# --------------------------------------------------

patient_crosswalk = (
    clinical_mapping[
        ["kit_id", "clinical_patient_id"]
    ]
    .merge(
        sensor_mapping,
        left_on="kit_id",
        right_on="Participant_ID",
        how="outer",
        indicator="sensor_match"
    )
    .drop(columns=["Participant_ID"])
)

patient_crosswalk = patient_crosswalk.merge(
    exacerbation_mapping[
        ["kit_id", "exacerbation_patient_id", "event_count"]
    ],
    on="kit_id",
    how="left"
)

print("\nFINAL PATIENT CROSSWALK")
print("=" * 50)
print("Total crosswalk rows:", len(patient_crosswalk))
print("Missing sensor matches:",
      patient_crosswalk["sensor_match"].ne("both").sum())
print("Missing clinical IDs:",
      patient_crosswalk["clinical_patient_id"].isna().sum())

print("\nParticipants with exacerbation records:",
      patient_crosswalk["event_count"].notna().sum())

display(patient_crosswalk.head(10))


# --------------------------------------------------
# 5. Check duplicate kit mappings
# --------------------------------------------------

print("\nDUPLICATE MAPPING CHECK")
print("=" * 50)

print("Duplicate clinical kit IDs:",
      clinical_mapping["kit_id"].duplicated().sum())

print("Duplicate sensor participant-country pairs:",
      sensor_mapping.duplicated(
          subset=["Participant_ID", "Country"]
      ).sum())

print("Duplicate crosswalk kit IDs:",
      patient_crosswalk["kit_id"].duplicated().sum())

### Step 7: Patient Crosswalk Validation

**Objective:** Verify that sensor participants, clinical records, and exacerbation events can be linked to the same patient records.

**Observed results:**

- 74 unique sensor participants.
- 74 unique clinical kit IDs.
- All 74 clinical kit IDs match sensor participant IDs.
- No participant has conflicting country values.
- No clinical file contains multiple numeric Patient-ID values.
- All 24 exacerbation kit IDs match the corresponding sensor participant IDs.
- No exacerbation kit ID maps to multiple numeric patient IDs.
- The final crosswalk contains 74 unique rows.
- No missing sensor matches or clinical patient IDs.
- No duplicate kit IDs in the crosswalk.

**Patient identification strategy:**

The common identifier across the datasets is the kit ID.

The Supabase `patients` table will use:
- `participant_id`: kit identifier (for example, GER-001).
- `country`: country obtained from the standardized sensor data.

The original numeric clinical and exacerbation patient identifiers will be retained as source metadata where appropriate.

**Interpretation:** All 74 participants have a consistent cross-dataset mapping. The patient crosswalk is ready to be used to create the database patient records.

In [ ]:
# --------------------------------------------------
# STEP 8: PREPARE PATIENT RECORDS
# --------------------------------------------------

patients_to_import = (
    patient_crosswalk[["kit_id", "Country"]]
    .rename(columns={
        "kit_id": "participant_id",
        "Country": "country"
    })
    .copy()
)

# Remove accidental whitespace
patients_to_import["participant_id"] = (
    patients_to_import["participant_id"].str.strip()
)

patients_to_import["country"] = (
    patients_to_import["country"].str.strip()
)

# Validation checks
assert len(patients_to_import) == 74
assert patients_to_import["participant_id"].nunique() == 74
assert patients_to_import["participant_id"].notna().all()
assert patients_to_import["country"].notna().all()

print("PATIENT IMPORT PREVIEW")
print("=" * 50)

print("Total patient records:", len(patients_to_import))
print("Unique participants:",
      patients_to_import["participant_id"].nunique())

print("\nCountry distribution:")
print(patients_to_import["country"].value_counts())

print("\nMissing values:")
print(patients_to_import.isna().sum())

print("\nDuplicate participant IDs:",
      patients_to_import["participant_id"].duplicated().sum())

display(patients_to_import.head(10))

### Step 8: Prepare Patient Records for Database Insertion

**Objective:** Prepare the patient master records using the validated patient crosswalk.

**Observed results:**

- Total patient records prepared: 74.
- Unique participant IDs: 74.
- Spain: 40 participants.
- Germany: 34 participants.
- Missing participant IDs: 0.
- Missing country values: 0.
- Duplicate participant IDs: 0.

**Prepared columns:**

| Column | Source |
|---|---|
| `participant_id` | Kit ID from the patient crosswalk |
| `country` | Country from the standardized sensor dataset |

**Interpretation:** The prepared dataframe satisfies the basic requirements of the Supabase `patients` table. All 74 patient records are ready for database insertion.

**Note:** This step only prepares and validates the records. No data has been inserted into Supabase yet.

In [ ]:
%pip install pandas pyarrow sqlalchemy "psycopg[binary]" python-dotenv

In [ ]:
# from pathlib import Path

# PROJECT_DIR = Path.cwd().parent

# env_path = PROJECT_DIR / ".env"

# env_path.touch(exist_ok=True)

# print("File created:", env_path)
# print("Exists:", env_path.exists())

In [ ]:
from pathlib import Path

env_path = PROJECT_DIR / ".env"

connection_string = input("Paste your Supabase connection string: ")

env_path.write_text(
    f'DATABASE_URL="{connection_string}"\n',
    encoding="utf-8"
)

print("Environment file saved:", env_path.exists())

In [ ]:
%pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import os

load_dotenv(PROJECT_DIR / ".env", override=True)

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

with engine.connect() as connection:
    result = connection.execute(text("SELECT NOW()"))
    print("Connected successfully!")
    print("Database time:", result.fetchone()[0])

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
env_path = PROJECT_DIR / ".env"

print("Project folder:", PROJECT_DIR)
print("Does .env exist?", env_path.exists())

In [ ]:
from getpass import getpass

connection_string = getpass("Paste your NEW Session Pooler URI: ")

env_path.write_text(
    f'DATABASE_URL="{connection_string}"\n',
    encoding="utf-8"
)

print(".env file created/updated successfully!")
print("File location:", env_path)

In [ ]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import os

load_dotenv(env_path, override=True)

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(
    DATABASE_URL,
    connect_args={"connect_timeout": 15}
)

with engine.connect() as connection:
    result = connection.execute(text("SELECT NOW()"))
    print("CONNECTED SUCCESSFULLY!")
    print("Database time:", result.fetchone()[0])

In [ ]:
from urllib.parse import urlparse
from dotenv import load_dotenv
import os

load_dotenv(env_path, override=True)

url = urlparse(os.getenv("DATABASE_URL"))

print("Username:", url.username)
print("Host:", url.hostname)
print("Port:", url.port)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(env_path, override=True)

url = os.getenv("DATABASE_URL")

print("Connection string exists:", bool(url))
print("Starts with postgresql://:", url.startswith("postgresql://"))
print("Contains [YOUR-PASSWORD]:", "[YOUR-PASSWORD]" in url)
print("Contains square brackets:", "[" in url or "]" in url)
print("Host section:", url.split("@")[-1] if "@" in url else "Missing @")

In [ ]:
from getpass import getpass
from urllib.parse import quote
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
env_path = PROJECT_DIR / ".env"

# Use the exact username shown in Supabase's Session Pooler URI
username = "postgres.odurqfhahaihiplienmz"

password = getpass("Enter your Supabase database password: ")

# URL-encode special characters in the password
encoded_password = quote(password, safe="")

host = "aws-0-ap-southeast-2.pooler.supabase.com"

DATABASE_URL = (
    f"postgresql://{username}:{encoded_password}@{host}:5432/postgres"
)

env_path.write_text(
    f'DATABASE_URL="{DATABASE_URL}"\n',
    encoding="utf-8"
)

print("Connection string saved successfully!")

In [ ]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import os

load_dotenv(env_path, override=True)

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(
    DATABASE_URL,
    connect_args={"connect_timeout": 15}
)

with engine.connect() as connection:
    result = connection.execute(text("SELECT NOW()"))
    print("CONNECTED SUCCESSFULLY!")
    print("Database time:", result.fetchone()[0])

In [ ]:
from urllib.parse import urlparse

saved_url = os.getenv("DATABASE_URL")
parsed = urlparse(saved_url)

print("Saved database username:", parsed.username)
print("Saved database host:", parsed.hostname)

In [ ]:
from getpass import getpass
from urllib.parse import quote
from pathlib import Path

password = getpass("Enter your NEW Supabase database password: ")

username = "postgres.odurqfhahaihiplienmz"
host = "aws-0-ap-southeast-2.pooler.supabase.com"

encoded_password = quote(password, safe="")

DATABASE_URL = (
    f"postgresql://{username}:{encoded_password}@{host}:5432/postgres"
)

env_path.write_text(
    f'DATABASE_URL="{DATABASE_URL}"\n',
    encoding="utf-8"
)

print("New connection string saved!")

In [ ]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv(env_path, override=True)

engine = create_engine(
    os.getenv("DATABASE_URL"),
    connect_args={"connect_timeout": 15}
)

with engine.connect() as connection:
    result = connection.execute(text("SELECT NOW()"))
    print("CONNECTED SUCCESSFULLY!")
    print("Database time:", result.fetchone()[0])

### Step: Verify Supabase Database Connection and Schema

A successful connection to the Supabase PostgreSQL database was established using SQLAlchemy and the Session Pooler connection.

The database was then inspected to:
- Verify the existence of all six expected tables.
- Check the current record count in each table.
- Confirm the database's initial state before importing standardized project data.

**Purpose:** Ensure that the database is accessible and its schema is ready before beginning data insertion.

**Note:** No records were inserted or modified during this verification step.

In [ ]:
from sqlalchemy import inspect, text

inspector = inspect(engine)

expected_tables = [
    "patients",
    "sensor_measurements",
    "clinical_assessments",
    "exacerbations",
    "predictions",
    "alerts"
]

existing_tables = inspector.get_table_names(schema="public")

print("DATABASE TABLE CHECK")
print("=" * 40)

for table in expected_tables:
    if table in existing_tables:
        with engine.connect() as connection:
            count = connection.execute(
                text(f'SELECT COUNT(*) FROM public."{table}"')
            ).scalar()

        print(f"{table}: EXISTS | {count} records")
    else:
        print(f"{table}: MISSING")

In [ ]:
print("PATIENT IMPORT VALIDATION")
print("=" * 40)

print("Total records:", len(patients_to_import))
print("Unique participants:", patients_to_import["participant_id"].nunique())
print("Missing values:")
print(patients_to_import.isnull().sum())

print("\nCountry distribution:")
print(patients_to_import["country"].value_counts())

print("\nPreview:")
display(patients_to_import.head())

In [ ]:
print("patient_crosswalk exists:", "patient_crosswalk" in globals())

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent

STANDARDIZED_DIR = PROJECT_DIR / "data" / "interim" / "standardized"
DATA_DIR = STANDARDIZED_DIR / "sensors"

print("Project directory:", PROJECT_DIR)
print("Sensor data directory exists:", DATA_DIR.exists())
print("Sensor files:")
print([f.name for f in DATA_DIR.glob("*.parquet")])

In [ ]:
sensor_files = list(DATA_DIR.glob("*.parquet"))

patient_data = []

for file in sensor_files:
    df = pd.read_parquet(file)

    patient_data.append(
        df[["Participant_ID", "Country"]].drop_duplicates()
    )

all_patients = pd.concat(patient_data, ignore_index=True)

patient_crosswalk = (
    all_patients
    .drop_duplicates(subset="Participant_ID")
    .rename(columns={"Participant_ID": "kit_id"})
    .reset_index(drop=True)
)

print("Unique patients:", len(patient_crosswalk))
print("Unique kit IDs:", patient_crosswalk["kit_id"].nunique())
print("\nCountry distribution:")
print(patient_crosswalk["Country"].value_counts())
print("\nMissing values:")
print(patient_crosswalk.isnull().sum())

In [ ]:
patients_to_import = (
    patient_crosswalk[["kit_id", "Country"]]
    .rename(columns={
        "kit_id": "participant_id",
        "Country": "country"
    })
    .copy()
)

patients_to_import["participant_id"] = (
    patients_to_import["participant_id"].str.strip()
)

patients_to_import["country"] = (
    patients_to_import["country"].str.strip()
)

print("PATIENT IMPORT VALIDATION")
print("=" * 40)
print("Total records:", len(patients_to_import))
print("Unique participants:", patients_to_import["participant_id"].nunique())
print("Missing values:")
print(patients_to_import.isnull().sum())

display(patients_to_import.head())

In [ ]:
from sqlalchemy import text

patient_records = patients_to_import.to_dict(orient="records")

insert_query = text("""
    INSERT INTO public.patients (participant_id, country)
    VALUES (:participant_id, :country)
    ON CONFLICT (participant_id) DO NOTHING
""")

with engine.begin() as connection:
    before_count = connection.execute(
        text("SELECT COUNT(*) FROM public.patients")
    ).scalar()

    connection.execute(insert_query, patient_records)

    after_count = connection.execute(
        text("SELECT COUNT(*) FROM public.patients")
    ).scalar()

print("PATIENT IMPORT RESULTS")
print("=" * 40)
print("Records before import:", before_count)
print("Records after import:", after_count)
print("New records inserted:", after_count - before_count)

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT participant_id, country
        FROM public.patients
        ORDER BY participant_id
        LIMIT 10
    """))

    patient_preview = pd.DataFrame(
        result.fetchall(),
        columns=result.keys()
    )

    total_patients = connection.execute(
        text("SELECT COUNT(*) FROM public.patients")
    ).scalar()

print("Total patients in database:", total_patients)
display(patient_preview)

In [ ]:
sensor_files = sorted(DATA_DIR.glob("*.parquet"))

print("SENSOR DATASET INSPECTION")
print("=" * 60)

for file in sensor_files:
    df = pd.read_parquet(file)

    print(f"\nDataset: {file.stem}")
    print("Total records:", len(df))
    print("Total columns:", len(df.columns))
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isnull().sum().sum())

In [ ]:
# Retrieve patient IDs from Supabase
with engine.connect() as connection:
    patients_db = pd.read_sql(
        text("""
            SELECT id, participant_id, country
            FROM public.patients
        """),
        connection
    )

# Create participant ID -> database ID mapping
patient_id_map = dict(
    zip(
        patients_db["participant_id"],
        patients_db["id"]
    )
)

print("PATIENT ID LOOKUP VALIDATION")
print("=" * 45)

print("Patients retrieved:", len(patients_db))
print("Unique participant IDs:", len(patient_id_map))

# Check all sensor participants against the database
all_sensor_ids = set()

for file in sensor_files:
    df = pd.read_parquet(file)
    all_sensor_ids.update(df["Participant_ID"].dropna().unique())

missing_patients = all_sensor_ids - set(patient_id_map.keys())

print("Unique sensor participants:", len(all_sensor_ids))
print("Participants missing from database:", len(missing_patients))

if missing_patients:
    print("Missing IDs:", sorted(missing_patients))
else:
    print("SUCCESS: Every sensor participant has a database ID!")

In [ ]:
with engine.connect() as connection:

    columns = pd.read_sql(
        text("""
            SELECT
                column_name,
                data_type,
                is_nullable
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = 'sensor_measurements'
            ORDER BY ordinal_position
        """),
        connection
    )

    indexes = pd.read_sql(
        text("""
            SELECT indexname, indexdef
            FROM pg_indexes
            WHERE schemaname = 'public'
              AND tablename = 'sensor_measurements'
        """),
        connection
    )

print("SENSOR TABLE COLUMNS")
print("=" * 60)
display(columns)

print("\nSENSOR TABLE INDEXES")
print("=" * 60)
display(indexes)

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT indexname, indexdef
        FROM pg_indexes
        WHERE schemaname = 'public'
          AND tablename = 'sensor_measurements'
    """))

    for row in result:
        print(f"\nIndex: {row.indexname}")
        print(row.indexdef)

In [ ]:
import json
import pandas as pd

# Load the BRI dataset
bri_df = pd.read_parquet(DATA_DIR / "BRI.parquet")

# Select the first source record
row_position = 0
row = bri_df.iloc[row_position]

sensor = "BRI"

# Convert missing values and NumPy values into JSON-compatible Python values
def clean_value(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

# Identify columns that are metadata rather than measurements
metadata_columns = {
    "Timestamp",
    "Sensor",
    "Participant_ID",
    "Country",
    "index_validity",
    "data_validity"
}

# Extract measurement values
measurements = {
    column: clean_value(row[column])
    for column in bri_df.columns
    if column not in metadata_columns
}

# Preserve original validity flags
validity_flags = {
    flag: clean_value(row[flag])
    for flag in ["index_validity", "data_validity"]
    if flag in bri_df.columns
}

# Convert timestamp to timezone-aware UTC datetime
measured_at = pd.to_datetime(
    row["Timestamp"],
    utc=True
).to_pydatetime()

# Construct the database record
sample_record = {
    "patient_id": patient_id_map[row["Participant_ID"]],
    "sensor_type": sensor,
    "measured_at": measured_at,
    "measurements": measurements,
    "is_valid": None,
    "validity_flags": validity_flags,
    "source_record_id": f"{sensor}:{row_position}"
}

print("ONE-RECORD IMPORT PREVIEW")
print("=" * 50)

for key, value in sample_record.items():
    print(f"\n{key}:")
    print(value)

# Check that JSONB fields can be serialized safely
json.dumps(sample_record["measurements"], allow_nan=False)
json.dumps(sample_record["validity_flags"], allow_nan=False)

print("\nSUCCESS: Both JSONB fields are JSON-compatible.")

In [ ]:
import json
import pandas as pd

metadata_columns = {
    "Timestamp",
    "Sensor",
    "Participant_ID",
    "Country",
    "index_validity",
    "data_validity"
}

def clean_value(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

sensor_import_records = []
validation_results = []

for file in sensor_files:
    sensor = file.stem
    df = pd.read_parquet(file)

    start_count = len(sensor_import_records)

    for row_position, row in df.iterrows():

        participant = row["Participant_ID"]

        if participant not in patient_id_map:
            raise ValueError(f"Unknown participant: {participant}")

        measured_at = pd.to_datetime(
            row["Timestamp"],
            utc=True,
            errors="raise"
        ).to_pydatetime()

        measurements = {
            column: clean_value(row[column])
            for column in df.columns
            if column not in metadata_columns
        }

        validity_flags = {
            flag: clean_value(row[flag])
            for flag in ["index_validity", "data_validity"]
            if flag in df.columns
        }

        record = {
            "patient_id": int(patient_id_map[participant]),
            "sensor_type": sensor,
            "measured_at": measured_at,
            "measurements": measurements,
            "is_valid": None,
            "validity_flags": validity_flags,
            "source_record_id": f"{sensor}:{row_position}"
        }

        # Verify JSON compatibility
        json.dumps(record["measurements"], allow_nan=False)
        json.dumps(record["validity_flags"], allow_nan=False)

        sensor_import_records.append(record)

    validation_results.append({
        "sensor": sensor,
        "source_rows": len(df),
        "prepared_records": len(sensor_import_records) - start_count
    })

validation_df = pd.DataFrame(validation_results)

print("SENSOR TRANSFORMATION VALIDATION")
print("=" * 55)
display(validation_df)

print("Total source rows:", validation_df["source_rows"].sum())
print("Total prepared records:", len(sensor_import_records))
print(
    "Unique source record IDs:",
    len({r["source_record_id"] for r in sensor_import_records})
)
print(
    "Records with missing measurement values:",
    sum(
        any(value is None for value in r["measurements"].values())
        for r in sensor_import_records
    )
)

print("\nSUCCESS: All sensor records passed transformation checks.")

In [ ]:
# Inspect one record from each sensor
for sensor in ["BRI", "EI", "MI", "PEI", "PRVI", "SHRI", "SII", "SQI"]:

    record = next(
        r for r in sensor_import_records
        if r["sensor_type"] == sensor
    )

    print(f"\n{sensor}")
    print("-" * 40)
    print("Source ID:", record["source_record_id"])
    print("Measurements:", record["measurements"])
    print("Validity flags:", record["validity_flags"])


# Count actual null measurement values
null_measurement_values = sum(
    value is None
    for record in sensor_import_records
    for value in record["measurements"].values()
)

total_measurement_values = sum(
    len(record["measurements"])
    for record in sensor_import_records
)

print("\nMISSING-VALUE CHECK")
print("=" * 40)
print("Total measurement values:", total_measurement_values)
print("Null measurement values:", null_measurement_values)
print("Missing values preserved:", all(
    value is None or not pd.isna(value)
    for record in sensor_import_records
    for value in record["measurements"].values()
))

In [ ]:
# Count missing values in validity flags
null_validity_flags = sum(
    value is None
    for record in sensor_import_records
    for value in record["validity_flags"].values()
)

# Count missing values in measurements and validity flags
print("MISSING-VALUE BREAKDOWN")
print("=" * 45)
print("Null measurement values:", null_measurement_values)
print("Null validity flag values:", null_validity_flags)
print(
    "Combined null values:",
    null_measurement_values + null_validity_flags
)

print("\nDifference from earlier validation:")
print(
    29147 - (null_measurement_values + null_validity_flags)
)

In [ ]:
from sqlalchemy import text

with engine.connect() as connection:
    sensor_count_before = connection.execute(
        text("SELECT COUNT(*) FROM public.sensor_measurements")
    ).scalar()

print("Current sensor records in Supabase:", sensor_count_before)

In [ ]:
from sqlalchemy import text, bindparam
from sqlalchemy.dialects.postgresql import JSONB

BATCH_SIZE = 1000

insert_query = text("""
    INSERT INTO public.sensor_measurements (
        patient_id,
        sensor_type,
        measured_at,
        measurements,
        is_valid,
        validity_flags,
        source_record_id
    )
    VALUES (
        :patient_id,
        :sensor_type,
        :measured_at,
        :measurements,
        :is_valid,
        :validity_flags,
        :source_record_id
    )
    ON CONFLICT (
        patient_id,
        sensor_type,
        source_record_id
    )
    WHERE source_record_id IS NOT NULL
    DO NOTHING
""").bindparams(
    bindparam("measurements", type_=JSONB),
    bindparam("validity_flags", type_=JSONB)
)

total_records = len(sensor_import_records)
inserted_count = 0

print("Starting sensor import...")
print("=" * 50)

for start in range(0, total_records, BATCH_SIZE):

    batch = sensor_import_records[start:start + BATCH_SIZE]

    with engine.begin() as connection:
        result = connection.execute(insert_query, batch)

    batch_inserted = result.rowcount
    inserted_count += batch_inserted

    print(
        f"Processed {min(start + BATCH_SIZE, total_records):,}"
        f" / {total_records:,} | "
        f"Inserted in this batch: {batch_inserted:,}"
    )

print("\nIMPORT COMPLETED")
print("=" * 50)
print("Prepared records:", total_records)
print("Newly inserted records:", inserted_count)
print("Records skipped as duplicates:", total_records - inserted_count)

In [ ]:
with engine.connect() as connection:

    # Total sensor records
    total_count = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM public.sensor_measurements
        """)
    ).scalar()

    # Records grouped by sensor
    sensor_counts = pd.read_sql(
        text("""
            SELECT
                sensor_type,
                COUNT(*) AS record_count,
                COUNT(DISTINCT patient_id) AS unique_patients,
                COUNT(DISTINCT measured_at) AS unique_timestamps
            FROM public.sensor_measurements
            GROUP BY sensor_type
            ORDER BY sensor_type
        """),
        connection
    )

print("DATABASE VERIFICATION")
print("=" * 55)
print("Total sensor records stored:", total_count)

print("\nSENSOR-WISE DATABASE COUNTS")
display(sensor_counts)

In [ ]:
schema_query = text("""
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name IN (
          'clinical_assessments',
          'exacerbations'
      )
    ORDER BY table_name, ordinal_position
""")

with engine.connect() as connection:
    schema_df = pd.read_sql(schema_query, connection)

display(schema_df)

In [ ]:
# Close stale pooled connections
engine.dispose()

# Recreate the schema inspection query
schema_query = text("""
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name IN (
          'clinical_assessments',
          'exacerbations'
      )
    ORDER BY table_name, ordinal_position
""")

# Open a fresh connection
with engine.connect() as connection:
    connection.rollback()

    schema_df = pd.read_sql_query(
        schema_query,
        connection
    )

display(schema_df)

In [ ]:
from pathlib import Path
from sqlalchemy import text

# Check current clinical assessment count
with engine.connect() as connection:
    clinical_count_before = connection.execute(
        text("SELECT COUNT(*) FROM public.clinical_assessments")
    ).scalar()

# Locate source files inside the project
clinical_files = list(PROJECT_DIR.rglob("clinical_indices.csv"))
exacerbation_files = list(PROJECT_DIR.rglob("exacerbations.csv"))

print("CLINICAL IMPORT BASELINE")
print("=" * 45)
print("Existing clinical records:", clinical_count_before)

print("\nCLINICAL SOURCE FILES")
for file in clinical_files:
    print(file)

print("\nEXACERBATION SOURCE FILES")
for file in exacerbation_files:
    print(file)

In [ ]:
clinical_frames = []

for file in clinical_files:
    df = pd.read_csv(file)

    # Participant ID comes from the folder name
    df["kit_id"] = file.parent.name

    # Country comes from the country folder
    df["source_country"] = file.parent.parent.name

    clinical_frames.append(df)

clinical_df = pd.concat(
    clinical_frames,
    ignore_index=True
)

# Identify rows containing actual clinical information
identifier_columns = [
    "Patient-ID",
    "Timepoint",
    "kit_id",
    "source_country"
]

clinical_value_columns = [
    col for col in clinical_df.columns
    if col not in identifier_columns
]

clinical_nonempty = clinical_df[
    clinical_df[clinical_value_columns].notna().any(axis=1)
].copy()

print("CLINICAL DATA COMBINATION")
print("=" * 50)
print("Participant files:", len(clinical_files))
print("Combined rows:", len(clinical_df))
print("Combined columns:", len(clinical_df.columns))
print("Rows with clinical information:", len(clinical_nonempty))
print("Empty visit rows:", len(clinical_df) - len(clinical_nonempty))
print("Unique participants:", clinical_df["kit_id"].nunique())

print("\nVISIT DISTRIBUTION")
display(clinical_nonempty["Timepoint"].value_counts().sort_index())

print("\nCOUNTRY DISTRIBUTION")
display(clinical_df.groupby("source_country")["kit_id"].nunique())

print("\nSAMPLE NON-EMPTY RECORDS")
display(clinical_nonempty.head(5))

In [ ]:
# 1. Load the patient records already stored in Supabase
with engine.connect() as connection:
    patients_db = pd.read_sql_query(
        text("""
            SELECT id, participant_id, country
            FROM public.patients
        """),
        connection
    )

# 2. Check that every clinical kit_id exists in the patients table
clinical_check = clinical_nonempty.merge(
    patients_db,
    left_on="kit_id",
    right_on="participant_id",
    how="left",
    indicator=True
)

# 3. Check for missing or unmatched identifiers
print("PARTICIPANT MAPPING")
print("=" * 50)
print("Clinical participants:", clinical_nonempty["kit_id"].nunique())
print("Patients in database:", patients_db["participant_id"].nunique())
print("Unmatched clinical rows:", (clinical_check["_merge"] == "left_only").sum())
print("Missing kit IDs:", clinical_nonempty["kit_id"].isna().sum())

# 4. Check whether the country in the source folder matches Supabase
clinical_check["country_matches"] = (
    clinical_check["source_country"] == clinical_check["country"]
)

print("\nCOUNTRY VALIDATION")
print("=" * 50)
print("Country mismatches:", (~clinical_check["country_matches"]).sum())

# 5. Check for duplicate participant-visit combinations
duplicate_visits = clinical_nonempty[
    clinical_nonempty.duplicated(
        subset=["kit_id", "Timepoint"],
        keep=False
    )
].sort_values(["kit_id", "Timepoint"])

print("\nVISIT UNIQUENESS")
print("=" * 50)
print("Duplicate participant-visit rows:", len(duplicate_visits))

# 6. Check required fields
print("\nREQUIRED FIELD VALIDATION")
print("=" * 50)
print("Missing kit_id:", clinical_nonempty["kit_id"].isna().sum())
print("Missing Timepoint:", clinical_nonempty["Timepoint"].isna().sum())
print("Missing Visit Date:", clinical_nonempty["Visit Date"].isna().sum())

# 7. Display any issues for inspection
if len(duplicate_visits) > 0:
    display(duplicate_visits[["kit_id", "Timepoint", "Visit Date"]])

if (clinical_check["_merge"] == "left_only").any():
    display(
        clinical_check.loc[
            clinical_check["_merge"] == "left_only",
            ["kit_id", "source_country"]
        ].drop_duplicates()
    )

if (~clinical_check["country_matches"]).any():
    display(
        clinical_check.loc[
            ~clinical_check["country_matches"],
            ["kit_id", "source_country", "country"]
        ].drop_duplicates()
    )

In [ ]:
from pathlib import Path

migration_dir = PROJECT_DIR / "database" / "migrations"
migration_dir.mkdir(parents=True, exist_ok=True)

migration_sql = """
-- Add a source identifier to make clinical assessment imports idempotent.
ALTER TABLE public.clinical_assessments
ADD COLUMN IF NOT EXISTS source_record_id TEXT;

-- Prevent duplicate source records for the same patient.
CREATE UNIQUE INDEX IF NOT EXISTS idx_clinical_source_record
ON public.clinical_assessments (patient_id, source_record_id)
WHERE source_record_id IS NOT NULL;
"""

migration_file = migration_dir / "002_clinical_source_record_id.sql"
migration_file.write_text(migration_sql.strip() + "\n", encoding="utf-8")

print("Migration file created:")
print(migration_file)
print("\nSQL:")
print(migration_sql)

In [ ]:
with engine.connect() as connection:
    migration_check = pd.read_sql_query(
        text("""
            SELECT
                column_name,
                data_type
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = 'clinical_assessments'
              AND column_name = 'source_record_id'
        """),
        connection
    )

    index_check = connection.execute(
        text("""
            SELECT indexname
            FROM pg_indexes
            WHERE schemaname = 'public'
              AND tablename = 'clinical_assessments'
              AND indexname = 'idx_clinical_source_record'
        """)
    ).fetchall()

print("COLUMN VERIFICATION")
display(migration_check)

print("\nINDEX VERIFICATION")
print(index_check)

In [ ]:
with engine.connect() as connection:
    print(connection.execute(
        text("SELECT current_database(), current_schema(), current_user")
    ).fetchone())

In [ ]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT
                table_schema,
                table_name,
                column_name,
                data_type
            FROM information_schema.columns
            WHERE table_name = 'clinical_assessments'
            ORDER BY ordinal_position
        """)
    ).fetchall()

for row in result:
    print(row)

In [ ]:
with engine.connect() as connection:
    index_result = connection.execute(
        text("""
            SELECT indexname, indexdef
            FROM pg_indexes
            WHERE schemaname = 'public'
              AND tablename = 'clinical_assessments'
              AND indexname = 'idx_clinical_source_record'
        """)
    ).fetchall()

print("CLINICAL SOURCE INDEX")
for row in index_result:
    print(row)

In [ ]:
import pandas as pd
import numpy as np

# Work on a copy so the original combined DataFrame remains unchanged
clinical_to_import = clinical_nonempty.copy()

# 1. Map participant kit IDs to Supabase patient IDs
clinical_to_import["patient_id"] = (
    clinical_to_import["kit_id"].map(patient_id_map)
)

# 2. Map visit type
clinical_to_import["visit_type"] = clinical_to_import["Timepoint"]

# 3. Convert date-only source values to UTC timestamps
clinical_to_import["visit_date"] = pd.to_datetime(
    clinical_to_import["Visit Date"],
    utc=True,
    errors="coerce"
)

# 4. Create a deterministic source ID
clinical_to_import["source_record_id"] = (
    clinical_to_import["kit_id"].astype(str)
    + ":"
    + clinical_to_import["Timepoint"].astype(str)
)

# 5. Identify original source columns
# Retain all 35 original CSV columns in clinical_data.
original_clinical_columns = list(
    clinical_df.columns.drop(["kit_id", "source_country"])
)

# 6. Convert missing values and NumPy values into JSON-compatible values
def clean_json_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, np.generic):
        return value.item()

    return value

clinical_to_import["clinical_data"] = clinical_to_import[
    original_clinical_columns
].apply(
    lambda row: {
        column: clean_json_value(value)
        for column, value in row.items()
    },
    axis=1
)

print("CLINICAL TRANSFORMATION SUMMARY")
print("=" * 50)
print("Prepared records:", len(clinical_to_import))
print("Unique patients:", clinical_to_import["patient_id"].nunique())
print("Unique source IDs:", clinical_to_import["source_record_id"].nunique())
print("Missing patient IDs:", clinical_to_import["patient_id"].isna().sum())
print("Missing visit types:", clinical_to_import["visit_type"].isna().sum())
print("Missing visit dates:", clinical_to_import["visit_date"].isna().sum())
print(
    "Duplicate source IDs:",
    clinical_to_import["source_record_id"].duplicated().sum()
)

print("\nSAMPLE TRANSFORMED RECORD")
display(
    clinical_to_import[
        [
            "patient_id",
            "visit_type",
            "visit_date",
            "source_record_id",
            "clinical_data"
        ]
    ].head(2)
)

In [ ]:
# Inspect the first complete clinical record
sample_record = clinical_to_import.iloc[0]

print("SOURCE RECORD ID:", sample_record["source_record_id"])
print("PATIENT ID:", sample_record["patient_id"])
print("VISIT TYPE:", sample_record["visit_type"])
print("VISIT DATE:", sample_record["visit_date"])

print("\nCOMPLETE CLINICAL DATA")
for key, value in sample_record["clinical_data"].items():
    print(f"{key}: {value}")

# Count missing values across the prepared JSONB data
total_missing = sum(
    value is None
    for record in clinical_to_import["clinical_data"]
    for value in record.values()
)

print("\nJSONB VALIDATION")
print("Total JSON null values:", total_missing)
print(
    "Original fields per record:",
    len(sample_record["clinical_data"])
)

In [ ]:
from sqlalchemy import text, bindparam
from sqlalchemy.dialects.postgresql import JSONB

# Prepare records for insertion
clinical_records = []

for _, row in clinical_to_import.iterrows():
    clinical_records.append({
        "patient_id": int(row["patient_id"]),
        "visit_type": row["visit_type"],
        "visit_date": row["visit_date"].to_pydatetime(),
        "clinical_data": row["clinical_data"],
        "source_record_id": row["source_record_id"]
    })

# SQL insertion query
insert_query = text("""
    INSERT INTO public.clinical_assessments (
        patient_id,
        visit_type,
        visit_date,
        clinical_data,
        source_record_id
    )
    VALUES (
        :patient_id,
        :visit_type,
        :visit_date,
        :clinical_data,
        :source_record_id
    )
    ON CONFLICT (patient_id, source_record_id)
    WHERE source_record_id IS NOT NULL
    DO NOTHING
""").bindparams(
    bindparam("clinical_data", type_=JSONB)
)

# Insert in batches
batch_size = 100
total_inserted = 0

with engine.begin() as connection:
    for start in range(0, len(clinical_records), batch_size):
        batch = clinical_records[start:start + batch_size]

        result = connection.execute(insert_query, batch)
        total_inserted += result.rowcount

print("CLINICAL DATABASE INSERTION")
print("=" * 50)
print("Prepared records:", len(clinical_records))
print("Successfully inserted:", total_inserted)
print("Skipped as duplicates:", len(clinical_records) - total_inserted)

In [ ]:
with engine.connect() as connection:

    # 1. Total number of clinical assessments
    total = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM public.clinical_assessments
        """)
    ).scalar()

    # 2. Distribution by visit type
    visit_distribution = pd.read_sql_query(
        text("""
            SELECT
                visit_type,
                COUNT(*) AS record_count,
                COUNT(DISTINCT patient_id) AS unique_patients
            FROM public.clinical_assessments
            GROUP BY visit_type
            ORDER BY visit_type
        """),
        connection
    )

    # 3. Check duplicate source IDs
    duplicate_count = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM (
                SELECT patient_id, source_record_id
                FROM public.clinical_assessments
                GROUP BY patient_id, source_record_id
                HAVING COUNT(*) > 1
            ) duplicates
        """)
    ).scalar()

    # 4. Check missing required fields
    missing_fields = connection.execute(
        text("""
            SELECT
                COUNT(*) FILTER (WHERE patient_id IS NULL) AS missing_patient_id,
                COUNT(*) FILTER (WHERE visit_type IS NULL) AS missing_visit_type,
                COUNT(*) FILTER (WHERE visit_date IS NULL) AS missing_visit_date,
                COUNT(*) FILTER (WHERE clinical_data IS NULL) AS missing_clinical_data,
                COUNT(*) FILTER (WHERE source_record_id IS NULL) AS missing_source_id
            FROM public.clinical_assessments
        """)
    ).fetchone()

print("CLINICAL DATABASE VERIFICATION")
print("=" * 50)
print("Total records:", total)

print("\nVISIT DISTRIBUTION")
display(visit_distribution)

print("\nDUPLICATE VALIDATION")
print("Duplicate patient-source ID combinations:", duplicate_count)

print("\nREQUIRED FIELD VALIDATION")
print(missing_fields)

In [ ]:
with engine.connect() as connection:
    event_schema = pd.read_sql_query(
        text("""
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = 'exacerbations'
            ORDER BY ordinal_position
        """),
        connection
    )

display(event_schema)

In [ ]:
exacerbation_frames = []

for file in exacerbation_files:
    df = pd.read_csv(file)

    # Preserve participant identity from the folder structure
    df["kit_id"] = file.parent.name
    df["source_country"] = file.parent.parent.name

    # Preserve the source file name for traceability
    df["source_file"] = str(file.relative_to(PROJECT_DIR))

    exacerbation_frames.append(df)

# Combine all participant files
exacerbation_df = pd.concat(
    exacerbation_frames,
    ignore_index=True
)

print("EXACERBATION DATA COMBINATION")
print("=" * 50)
print("Participant files:", len(exacerbation_files))
print("Combined rows:", len(exacerbation_df))
print("Combined columns:", len(exacerbation_df.columns))
print("Unique participants:", exacerbation_df["kit_id"].nunique())

print("\nSEVERITY DISTRIBUTION")
display(exacerbation_df["Type"].value_counts(dropna=False))

print("\nCOUNTRY DISTRIBUTION")
display(
    exacerbation_df.groupby("source_country")["kit_id"].nunique()
)

print("\nSAMPLE EVENTS")
display(exacerbation_df.head(10))

In [ ]:
# 1. Match exacerbation participants with Supabase patients
exacerbation_check = exacerbation_df.merge(
    patients_db,
    left_on="kit_id",
    right_on="participant_id",
    how="left",
    indicator=True
)

# 2. Check folder ID against original Kit-id
kit_mismatches = (
    exacerbation_df["kit_id"].astype(str).str.strip()
    != exacerbation_df["Kit-id"].astype(str).str.strip()
)

# 3. Check numeric PatientId against clinical source IDs
clinical_id_map = (
    clinical_nonempty[["kit_id", "Patient-ID"]]
    .drop_duplicates()
    .set_index("kit_id")["Patient-ID"]
)

expected_patient_ids = exacerbation_df["kit_id"].map(clinical_id_map)

patient_id_mismatches = (
    pd.to_numeric(exacerbation_df["PatientId"], errors="coerce")
    != pd.to_numeric(expected_patient_ids, errors="coerce")
)

# 4. Check countries
country_mismatches = (
    exacerbation_check["source_country"]
    != exacerbation_check["country"]
)

# 5. Check event dates
parsed_dates = pd.to_datetime(
    exacerbation_df["Date"],
    errors="coerce"
)

# 6. Check exact duplicate source event rows
duplicate_events = exacerbation_df.duplicated(
    subset=["Date", "PatientId", "Kit-id", "Type"],
    keep=False
)

# 7. Validate severity
valid_severities = ["Moderate", "Severe"]
invalid_severity = ~exacerbation_df["Type"].isin(valid_severities)

# 8. Print results
print("PARTICIPANT MAPPING")
print("=" * 50)
print("Unique event participants:", exacerbation_df["kit_id"].nunique())
print("Unmatched event rows:", (exacerbation_check["_merge"] == "left_only").sum())

print("\nIDENTIFIER VALIDATION")
print("=" * 50)
print("Kit ID mismatches:", kit_mismatches.sum())
print("Numeric PatientId mismatches:", patient_id_mismatches.sum())
print("Country mismatches:", country_mismatches.sum())

print("\nEVENT VALIDATION")
print("=" * 50)
print("Missing dates:", parsed_dates.isna().sum())
print("Missing PatientId:", exacerbation_df["PatientId"].isna().sum())
print("Missing Kit-id:", exacerbation_df["Kit-id"].isna().sum())
print("Missing severity:", exacerbation_df["Type"].isna().sum())
print("Invalid severity values:", invalid_severity.sum())
print("Rows involved in exact duplicate events:", duplicate_events.sum())

# Display any issues
if kit_mismatches.any():
    display(exacerbation_df.loc[kit_mismatches])

if patient_id_mismatches.any():
    display(exacerbation_df.loc[patient_id_mismatches])

if country_mismatches.any():
    display(exacerbation_check.loc[
        country_mismatches,
        ["kit_id", "source_country", "country"]
    ])

if duplicate_events.any():
    display(exacerbation_df.loc[duplicate_events])

In [ ]:
# Work on a separate copy
exacerbation_to_import = exacerbation_df.copy()

# 1. Map kit IDs to Supabase patient IDs
exacerbation_to_import["patient_id"] = (
    exacerbation_to_import["kit_id"].map(patient_id_map)
)

# 2. Convert date-only values to UTC timestamps
exacerbation_to_import["event_timestamp"] = pd.to_datetime(
    exacerbation_to_import["Date"],
    utc=True,
    errors="coerce"
)

# 3. Map event severity
exacerbation_to_import["severity"] = (
    exacerbation_to_import["Type"].str.strip()
)

# 4. Generate deterministic source event IDs
# The source file and original row position together identify the record.
exacerbation_to_import["source_event_id"] = (
    exacerbation_to_import["source_file"].astype(str)
    + ":"
    + exacerbation_to_import.groupby("source_file").cumcount().astype(str)
)

# 5. Preserve the original four CSV fields in JSONB
original_event_columns = ["Date", "PatientId", "Kit-id", "Type"]

def clean_event_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value

exacerbation_to_import["event_data"] = (
    exacerbation_to_import[original_event_columns].apply(
        lambda row: {
            column: clean_event_value(value)
            for column, value in row.items()
        },
        axis=1
    )
)

# 6. Summary
print("EXACERBATION TRANSFORMATION SUMMARY")
print("=" * 50)
print("Prepared records:", len(exacerbation_to_import))
print("Unique patients:", exacerbation_to_import["patient_id"].nunique())
print("Unique source IDs:", exacerbation_to_import["source_event_id"].nunique())
print("Missing patient IDs:", exacerbation_to_import["patient_id"].isna().sum())
print("Missing timestamps:", exacerbation_to_import["event_timestamp"].isna().sum())
print("Missing severity:", exacerbation_to_import["severity"].isna().sum())
print(
    "Duplicate source IDs:",
    exacerbation_to_import["source_event_id"].duplicated().sum()
)

print("\nSEVERITY DISTRIBUTION")
display(exacerbation_to_import["severity"].value_counts())

print("\nSAMPLE TRANSFORMED EVENTS")
display(
    exacerbation_to_import[
        [
            "patient_id",
            "event_timestamp",
            "severity",
            "source_event_id",
            "event_data"
        ]
    ].head(5)
)

In [ ]:
from sqlalchemy import text, bindparam
from sqlalchemy.dialects.postgresql import JSONB

# 1. Prepare database-ready records
exacerbation_records = []

for _, row in exacerbation_to_import.iterrows():
    exacerbation_records.append({
        "patient_id": int(row["patient_id"]),
        "event_timestamp": row["event_timestamp"].to_pydatetime(),
        "severity": row["severity"],
        "source_event_id": row["source_event_id"],
        "event_data": row["event_data"]
    })

# 2. Define insertion query
insert_event_query = text("""
    INSERT INTO public.exacerbations (
        patient_id,
        event_timestamp,
        severity,
        source_event_id,
        event_data
    )
    VALUES (
        :patient_id,
        :event_timestamp,
        :severity,
        :source_event_id,
        :event_data
    )
    ON CONFLICT (patient_id, source_event_id)
    WHERE source_event_id IS NOT NULL
    DO NOTHING
""").bindparams(
    bindparam("event_data", type_=JSONB)
)

# 3. Insert in batches
batch_size = 100
total_events_inserted = 0

with engine.begin() as connection:
    for start in range(0, len(exacerbation_records), batch_size):
        batch = exacerbation_records[start:start + batch_size]

        result = connection.execute(insert_event_query, batch)
        total_events_inserted += result.rowcount

# 4. Print summary
print("EXACERBATION DATABASE INSERTION")
print("=" * 50)
print("Prepared records:", len(exacerbation_records))
print("Successfully inserted:", total_events_inserted)
print(
    "Skipped as duplicates:",
    len(exacerbation_records) - total_events_inserted
)

In [ ]:
with engine.connect() as connection:

    # 1. Total event count
    total_events = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM public.exacerbations
        """)
    ).scalar()

    # 2. Severity distribution
    severity_distribution = pd.read_sql_query(
        text("""
            SELECT
                severity,
                COUNT(*) AS event_count,
                COUNT(DISTINCT patient_id) AS unique_patients
            FROM public.exacerbations
            GROUP BY severity
            ORDER BY severity
        """),
        connection
    )

    # 3. Duplicate source IDs
    duplicate_events = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM (
                SELECT patient_id, source_event_id
                FROM public.exacerbations
                GROUP BY patient_id, source_event_id
                HAVING COUNT(*) > 1
            ) duplicates
        """)
    ).scalar()

    # 4. Required field validation
    event_missing_fields = connection.execute(
        text("""
            SELECT
                COUNT(*) FILTER (WHERE patient_id IS NULL) AS missing_patient_id,
                COUNT(*) FILTER (WHERE event_timestamp IS NULL) AS missing_timestamp,
                COUNT(*) FILTER (WHERE severity IS NULL) AS missing_severity,
                COUNT(*) FILTER (WHERE source_event_id IS NULL) AS missing_source_id,
                COUNT(*) FILTER (WHERE event_data IS NULL) AS missing_event_data
            FROM public.exacerbations
        """)
    ).fetchone()

    # 5. Check event date range
    event_date_range = connection.execute(
        text("""
            SELECT
                MIN(event_timestamp) AS earliest_event,
                MAX(event_timestamp) AS latest_event
            FROM public.exacerbations
        """)
    ).fetchone()

print("EXACERBATION DATABASE VERIFICATION")
print("=" * 50)
print("Total events:", total_events)

print("\nSEVERITY DISTRIBUTION")
display(severity_distribution)

print("\nDUPLICATE VALIDATION")
print("Duplicate patient-source ID combinations:", duplicate_events)

print("\nREQUIRED FIELD VALIDATION")
print(event_missing_fields)

print("\nEVENT DATE RANGE")
print(event_date_range)

In [ ]:
from sqlalchemy import text
import pandas as pd

with engine.connect() as connection:

    # Total records in each table
    table_counts = pd.read_sql_query(
        text("""
            SELECT 'patients' AS table_name, COUNT(*) AS total_records
            FROM public.patients

            UNION ALL

            SELECT 'sensor_measurements', COUNT(*)
            FROM public.sensor_measurements

            UNION ALL

            SELECT 'clinical_assessments', COUNT(*)
            FROM public.clinical_assessments

            UNION ALL

            SELECT 'exacerbations', COUNT(*)
            FROM public.exacerbations

            UNION ALL

            SELECT 'predictions', COUNT(*)
            FROM public.predictions

            UNION ALL

            SELECT 'alerts', COUNT(*)
            FROM public.alerts
        """),
        connection
    )

    # Check for records whose patient IDs do not exist
    foreign_key_checks = pd.read_sql_query(
        text("""
            SELECT
                'sensor_measurements' AS table_name,
                COUNT(*) AS orphan_records
            FROM public.sensor_measurements s
            LEFT JOIN public.patients p
                ON s.patient_id = p.id
            WHERE p.id IS NULL

            UNION ALL

            SELECT
                'clinical_assessments',
                COUNT(*)
            FROM public.clinical_assessments c
            LEFT JOIN public.patients p
                ON c.patient_id = p.id
            WHERE p.id IS NULL

            UNION ALL

            SELECT
                'exacerbations',
                COUNT(*)
            FROM public.exacerbations e
            LEFT JOIN public.patients p
                ON e.patient_id = p.id
            WHERE p.id IS NULL

            UNION ALL

            SELECT
                'predictions',
                COUNT(*)
            FROM public.predictions pr
            LEFT JOIN public.patients p
                ON pr.patient_id = p.id
            WHERE p.id IS NULL

            UNION ALL

            SELECT
                'alerts',
                COUNT(*)
            FROM public.alerts a
            LEFT JOIN public.patients p
                ON a.patient_id = p.id
            WHERE p.id IS NULL
        """),
        connection
    )

print("FINAL DATABASE TABLE AUDIT")
print("=" * 60)

print("\nTABLE RECORD COUNTS")
display(table_counts)

print("\nFOREIGN KEY INTEGRITY")
display(foreign_key_checks)

print("\nTOTAL RECORDS ACROSS ALL TABLES:")
print(table_counts["total_records"].sum())

### Final Database Audit — Successful

The final audit of the ExacerbAlert Supabase PostgreSQL database was completed successfully.

**Final table counts:**

| Database table | Records |
|---|---:|
| patients | 74 |
| sensor_measurements | 91,794 |
| clinical_assessments | 224 |
| exacerbations | 39 |
| predictions | 0 |
| alerts | 0 |
| **Total** | **92,131** |

**Foreign-key integrity:** All five patient-linked tables were checked against the `patients` table. No orphan records were found.

**Interpretation:**

- All expected patient, sensor, clinical, and exacerbation records are present.
- All patient references resolve to existing patient records.
- Predictions and alerts remain empty intentionally, pending model and application integration.

**Conclusion:** The database import process and the validations performed during this stage were successful. The database is ready for the next stage of project development, subject to further schema, security, and application-level testing.